In [1]:
import pandas as pd
import numpy
import numpy as np

df_raw = pd.read_csv('../Data/Raw/zomato.csv')
print(f"Dimensi dataset: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

display(df_raw.head())

df_raw.info()

Dimensi dataset: 51717 baris, 17 kolom


,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


<class 'pandas.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   url                          51717 non-null  str  
 1   address                      51717 non-null  str  
 2   name                         51717 non-null  str  
 3   online_order                 51717 non-null  str  
 4   book_table                   51717 non-null  str  
 5   rate                         43942 non-null  str  
 6   votes                        51717 non-null  int64
 7   phone                        50509 non-null  str  
 8   location                     51696 non-null  str  
 9   rest_type                    51490 non-null  str  
 10  dish_liked                   23639 non-null  str  
 11  cuisines                     51672 non-null  str  
 12  approx_cost(for two people)  51371 non-null  str  
 13  reviews_list                 51717 non-null  str  
 14  m

#### Data Understanding:
##### Dataset ini berisi informasi restoran di Bangalore wilayah india yang terdapat di platform Zomato, seperti nama restoran, lokasi, rating, jenis makanan, dan harga.

##### Dataset memiliki sekitar 50.000+ baris dan 17 kolom.


In [2]:
def clean_general(df):
    data = df.copy()

    data = data.drop_duplicates()

    data = data.rename(columns={
        'approx_cost(for two people)': 'cost_for_two',
        'listed_in(type)': 'service_type',
    })

    data = data.drop(columns=['url', 'address', 'phone', 'menu_item', 'reviews_list', 'dish_liked', 'listed_in(city)'])

    data = data.dropna(subset=['cuisines', 'location', 'rest_type'])

    string_cols = data.select_dtypes(include=['str']).columns
    for col in string_cols:
        data[col] = data[col].str.title().str.strip()

    for col in data.columns:
        if any(keyword in col.lower() for keyword in ['cost', 'votes', 'rate']):
            pass

    return data

df_general_cleaning= clean_general(df_raw)







di clean general,saya menghapus kolom yg tidak relevan dgn analisis saya karena ada beberapa kolom yg nilai nya abstract contohnya menu item yg di isi kurung siku saja,dan untuk kolom yg lain seperti dish liked saya hapus karena banyak missing value nya dan saya tidak mau memakai nya karena sudah ada kolom cuisines.untuk handle missing value saya hapus kolom cuisines,location,dan,rest type karena data nya sangat dikit yg missing value,selanjutnya saya membuat standarisasi teks menggunakan title untuk kolom string,saya menghapus duplikat,memberikan rename terhadap kolom,dan validasi tipe data jika ketemu cost,votes,rate akan diubah menjadi integer di function selanjutnya yaitu spesific cleaning


In [3]:
df_general_cleaning.head()


,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost_for_two,service_type
0,Jalsa,Yes,Yes,4.1/5,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800,Buffet
1,Spice Elephant,Yes,No,4.1/5,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800,Buffet
2,San Churro Cafe,Yes,No,3.8/5,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800,Buffet
3,Addhuri Udupi Bhojana,No,No,3.7/5,88,Banashankari,Quick Bites,"South Indian, North Indian",300,Buffet
4,Grand Village,No,No,3.8/5,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600,Buffet


In [4]:
df_general_cleaning.info()

<class 'pandas.DataFrame'>
Index: 51466 entries, 0 to 51716
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   name          51466 non-null  str  
 1   online_order  51466 non-null  str  
 2   book_table    51466 non-null  str  
 3   rate          43780 non-null  str  
 4   votes         51466 non-null  int64
 5   location      51466 non-null  str  
 6   rest_type     51466 non-null  str  
 7   cuisines      51466 non-null  str  
 8   cost_for_two  51148 non-null  str  
 9   service_type  51466 non-null  str  
dtypes: int64(1), str(9)
memory usage: 8.5 MB


In [5]:
def clean_spesific(df_general_cleaning):
    data = df_general_cleaning.copy()

    if 'rate' in data.columns:
        data['rate'] = data['rate'].astype(str).str.replace('/5', '', regex=False)
        data['rate'] = data['rate'].str.lower().replace(['new', '-', 'nan'], np.nan)
        data['rate'] = data['rate'].astype(float)
        data = data.dropna(subset=['rate'])

    if 'cost_for_two' in data.columns:
        data['cost_for_two'] = data['cost_for_two'].astype(str).str.replace(',', '', regex=False)
        data['cost_for_two'] = data['cost_for_two'].str.lower().replace('nan', np.nan)
        data['cost_for_two'] = data['cost_for_two'].astype(float)
        data['cost_for_two'] = data['cost_for_two'].fillna(data['cost_for_two'].median())

    if 'votes' in data.columns:
        data['votes'] = pd.to_numeric(data['votes'], errors='coerce').fillna(0).astype(int)

    if 'location' in data.columns:
        data.loc[data['location'].str.contains('koramangala', case=False, na=False), 'location'] = 'Koramangala'

    return data

1.saya membersihkan kolom rating ,pertama saya memotong string "/5" karena masalah nya sebelum cleaning ,rating itu bentuk nya seperti ini 4.1/5,saya mau memotong 5 nya supaya enak dilihat.langkah kedua ada masalah yaitu nilai rate ada nilai nya string "new" bukan rating biasa,jadi saya mengubah nya menjadi nilai kosong setiap ketemu new.langkah ketiga saya mengubah kolom rating menjadi tipe data float,langkah terakhir saya menghapus baris data yg hilang di kolom rate ,alasan saya langsung drop karena jika saya melakukan imputasi data ,ini tidak akan menggambarkan kondisi tingkat kepuasan pelanggan yg sebenarnya jadi saya tidak mau merekayasa kualitas restoran.2.saya membersihkan kolom cost_for_two,langkah pertama saya mengubah 1,200 menjadi 1200 jadi saya replace koma nya karena Python tidak bisa mengonversi teks yang ada komanya menjadi angka.step kedua sama seperti kolom rate yg saya lakukan ,yaitu mengubah string "nan" menjadi nilai kosong,lalu langkah ketiga mengubah tipe data,langkah keempat saya mengimputasi median terhadap nilai yg hilang karena data nya dibawah 5 persen dan saya tidak mau kehilangan banyak informasi di kolom lain juga makanya saya pertahankan,saya memilih imputasi median karena lebih tahan terhadap nilai outlier.langkah ke lima saya menstandarisasi data location yg memiliki satu entitas tetapi ada banyak jenis ,jadi saya mau standarisai hal itu agar dianggap satu entitas saja.

In [6]:
df_clean = clean_spesific(df_general_cleaning)

print(f"Dimensi setelah Full Cleaning: {df_clean.shape[0]} baris, {df_clean.shape[1]} kolom")
display(df_clean.head())
df_clean.info()

Dimensi setelah Full Cleaning: 41505 baris, 10 kolom


,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost_for_two,service_type
0,Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,Buffet
1,Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,Buffet
2,San Churro Cafe,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800.0,Buffet
3,Addhuri Udupi Bhojana,No,No,3.7,88,Banashankari,Quick Bites,"South Indian, North Indian",300.0,Buffet
4,Grand Village,No,No,3.8,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600.0,Buffet


<class 'pandas.DataFrame'>
Index: 41505 entries, 0 to 51716
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          41505 non-null  str    
 1   online_order  41505 non-null  str    
 2   book_table    41505 non-null  str    
 3   rate          41505 non-null  float64
 4   votes         41505 non-null  int64  
 5   location      41505 non-null  str    
 6   rest_type     41505 non-null  str    
 7   cuisines      41505 non-null  str    
 8   cost_for_two  41505 non-null  float64
 9   service_type  41505 non-null  str    
dtypes: float64(2), int64(1), str(7)
memory usage: 6.5 MB


In [7]:
df_clean['location'].unique()

<ArrowStringArray>
[                 'Banashankari',                  'Basavanagudi',
                   'Mysore Road',                     'Jayanagar',
            'Kumaraswamy Layout',          'Rajarajeshwari Nagar',
                   'Vijay Nagar',                   'Uttarahalli',
                      'Jp Nagar',               'South Bangalore',
                   'City Market',             'Bannerghatta Road',
                           'Btm',               'Kanakapura Road',
                  'Bommanahalli',               'Electronic City',
                 'Wilson Garden',                  'Shanti Nagar',
                   'Koramangala',                 'Richmond Road',
                           'Hsr',                     'Bellandur',
                 'Sarjapur Road',                  'Marathahalli',
                    'Whitefield',                'East Bangalore',
              'Old Airport Road',                   'Indiranagar',
                   'Frazer Town',          

In [8]:
def feature_engginering(df_clean):
    data =df_clean.copy()

    data['cost_category'] = pd.qcut(
        data['cost_for_two'],
        q=3,
        labels=['Murah', 'Sedang', 'Mahal']
    )

    data['popularity_score'] = data['rate'] * np.log1p(data['votes'])

    data = data[data['votes'] > 0]

    return data

saya membuat kolom baru yaitu cost_category dari kolom cost for two untuk membuat fitur kategori harga dgn 3 nilai yaitu murah,sedang,dan mahal,selanjutnya saya membuat fitur baru lagi yaitu popularity score dari kolom rate dan votes sebelumnya ,tujuanya mengukur popularitas restoran

In [9]:
df_final = feature_engginering(df_clean)

display(df_final[['name', 'rate', 'votes', 'cost_category', 'popularity_score']].head())

,name,rate,votes,cost_category,popularity_score
0,Jalsa,4.1,775,Mahal,27.282025
1,Spice Elephant,4.1,787,Mahal,27.344942
2,San Churro Cafe,3.8,918,Mahal,25.928487
3,Addhuri Udupi Bhojana,3.7,88,Murah,16.607955
4,Grand Village,3.8,166,Sedang,19.448376


In [10]:
df_final

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost_for_two,service_type,cost_category,popularity_score
0,Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,Buffet,Mahal,27.282025
1,Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,Buffet,Mahal,27.344942
2,San Churro Cafe,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800.0,Buffet,Mahal,25.928487
3,Addhuri Udupi Bhojana,No,No,3.7,88,Banashankari,Quick Bites,"South Indian, North Indian",300.0,Buffet,Murah,16.607955
4,Grand Village,No,No,3.8,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600.0,Buffet,Sedang,19.448376
...,...,...,...,...,...,...,...,...,...,...,...,...
51709,The Farm House Bar N Grill,No,No,3.7,34,Whitefield,"Casual Dining, Bar","North Indian, Continental",800.0,Pubs And Bars,Mahal,13.154788
51711,Bhagini,No,No,2.5,81,Whitefield,"Casual Dining, Bar","Andhra, South Indian, Chinese, North Indian",800.0,Pubs And Bars,Mahal,11.016798
51712,Best Brews - Four Points By Sheraton Bengaluru...,No,No,3.6,27,Whitefield,Bar,Continental,1500.0,Pubs And Bars,Mahal,11.995936
51715,Chime - Sheraton Grand Bengaluru Whitefield Ho...,No,Yes,4.3,236,"Itpl Main Road, Whitefield",Bar,Finger Food,2500.0,Pubs And Bars,Mahal,23.512659


In [11]:
def generate_bussines_use_cases(df_final):
    df = df_final.copy()

    loc_analysis = df.groupby('location').agg(
        total_resto=('name', 'count'),
        avg_rating=('rate', 'mean'),
        total_votes=('votes', 'sum')
    ).reset_index().sort_values('total_votes', ascending=False)

    price_analysis = df.groupby('cost_category', observed=True).agg(
        total_resto=('name', 'count'),
        avg_rating=('rate', 'mean'),
        avg_popularity=('popularity_score', 'mean')
    ).reset_index()

    online_analysis = df.groupby('online_order').agg(
        total_resto=('name', 'count'),
        avg_votes=('votes', 'mean'),
        avg_popularity=('popularity_score', 'mean')
    ).reset_index()

    return loc_analysis, price_analysis, online_analysis




Di blok kode ini, kita bikin satu *function* bernama `generate_bussines_use_cases` buat ngeringkas puluhan ribu data mentah kita jadi 3 tabel metrik utama. Intinya, kita pakai fungsi `.groupby()` buat nge-grup data berdasarkan:

1. **Lokasi (`loc_analysis`)**: Buat nyari tau area mana yang *traffic* pelanggannya paling tinggi.
2. **Harga (`price_analysis`)**: Buat ngecek korelasi apakah harga yang lebih mahal ngaruh ke rating kepuasan yang lebih bagus.
3. **Online Order (`online_analysis`)**: Buat ngebandingin rata-rata *votes* antara resto yang go-digital (pake online order) vs resto murni *offline*.


In [12]:
df_loc, df_price, df_online = generate_bussines_use_cases(df_final)

print("=== HASIL USE CASE 1: IDENTIFIKASI LOKASI STRATEGIS ===")
display(df_loc)
print("\n=== HASIL USE CASE 2: ANALISIS STRATEGI HARGA ===")
display(df_price)
print("\n=== HASIL USE CASE 3: DAMPAK ONLINE ORDER ===")
display(df_online)

=== HASIL USE CASE 1: IDENTIFIKASI LOKASI STRATEGIS ===


,location,total_resto,avg_rating,total_votes
38,Koramangala,6665,3.882146,4278072
26,Indiranagar,1842,3.828013,1194740
9,Btm,3902,3.573578,612689
11,Church Street,546,3.992125,594979
32,Jp Nagar,1701,3.677425,578010
...,...,...,...,...
22,Hebbal,8,3.600000,450
39,Kr Puram,10,3.540000,277
82,Yelahanka,5,3.640000,189
49,Nagarbhavi,1,3.400000,10



=== HASIL USE CASE 2: ANALISIS STRATEGI HARGA ===


,cost_category,total_resto,avg_rating,avg_popularity
0,Murah,13884,3.568302,12.643039
1,Sedang,14703,3.624804,15.859181
2,Mahal,12899,3.934383,22.624542



=== HASIL USE CASE 3: DAMPAK ONLINE ORDER ===


,online_order,total_resto,avg_votes,avg_popularity
0,No,14399,368.655045,15.631748
1,Yes,27087,344.337099,17.553289


In [13]:
# Ekspor data master (df_final) dan tabel use case ke Excel
with pd.ExcelWriter('../Data/processed/Zomato_Final_Analysis.xlsx') as writer:
    # Sheet 1: DATA MASTER (Semua 41k baris untuk keperluan Drag & Drop Pivot Table)
    df_final.to_excel(writer, sheet_name='Data_Master_Full', index=False)

    # Sheet 2, 3, 4: Hasil Agregasi yang sudah kita buat
    df_loc.to_excel(writer, sheet_name='Lokasi_Strategis', index=False)
    df_price.to_excel(writer, sheet_name='Analisis_Harga', index=False)
    df_online.to_excel(writer, sheet_name='Dampak_Online', index=False)

print("Export selesai! Buka file Excel, lalu gunakan sheet 'Data_Master_Full' untuk membuat Pivot Table sesukamu.")

Export selesai! Buka file Excel, lalu gunakan sheet 'Data_Master_Full' untuk membuat Pivot Table sesukamu.


![Grafik Lokasi](../Image/bussinescase1.png)

chart ini untuk mengetahui lokasi mana yg punya total interaksi pelanggan yg tertinggi (votes),berdasarkan insight dari data tersebut menunjukan bahwa daerah koragmala begitu menguasai daerah bangalore.

![Grafik Lokasi](../Image/bussinescase2.png)

di chart ini kita ingin mencari tahu apakah kategori harga rendah atau tinggi itu menentukan pengalaman makan disana data ini menunjukan bahwa kategori harga mahal mempunyai rata rata rating yg tinggi karena penduduk sana sudah mendapatkan hal yg sesuai dgn keinginan nya ketika membeli makanan yg mahal yaitu pengalaman baik nya.

![Grafik Lokasi](../Image/bussinescase3.png)

Tujuan kita di sini adalah melihat perbandingan rata-rata votes antara restoran yang melayani online dan yang tidak.instight dari data ini menunjukan bahwa dgn  menambahkan fitur online di sebuah bisnis akan berpotensi mendapatkan interaksi pelanggan yg cukup tinggi,karena orang dari manapun berasal dapat melihat bisnis anda ketimbang yg tidak memakai fitur online